In [1]:
import numpy as np
from tcn_weights import *

# ============================================================
# tanh approximation (5th‑order, SAME as HLS)
# ============================================================
def tanh_approx(x):
    x = np.clip(x, -1.0, 1.0)
    x2 = x * x
    x3 = x2 * x
    x5 = x3 * x2
    return x - x3 / 3.0 + x5 / 5.0

# ============================================================
# 1D causal convolution (layer 1)
# ============================================================
def conv1d(in_win, W, B, out_ch):
    out = np.zeros((TCN_WIN, out_ch), dtype=np.float32)

    for t in range(TCN_WIN):
        for c in range(out_ch):
            acc = B[c]
            for k in range(TCN_K):
                ti = t - k
                if ti >= 0:
                    for i in range(in_win.shape[1]):
                        acc += W[c, i, k] * in_win[ti, i]
            out[t, c] = tanh_approx(acc)
    return out

# ============================================================
# Dense output (last timestep only)
# ============================================================
def dense_out(x):
    y = np.zeros(N_OUTPUT, dtype=np.float32)
    for o in range(N_OUTPUT):
        acc = B_out[o]
        for i in range(TCN_CH2):
            acc += W_out[o, i] * x[i]
        y[o] = acc
    return y

# ============================================================
# SRT taps
# ============================================================
def srt_taps(y):
    v = np.uint32(0)
    for i in range(N_OUTPUT):
        bits = np.frombuffer(y[i].tobytes(), dtype=np.uint32)[0]
        v |= (bits & 0x3FF) << (10 * i)
    return v

# ============================================================
# ARX
# ============================================================
def arx(x):
    a = ((x << 5) | (x >> 27)) & 0xFFFFFFFF
    b = (x + 0x9E3779B9) & 0xFFFFFFFF
    return np.uint32(a ^ b)

# ============================================================
# TCN RNG class (equivalent to tcn_rng_top)
# ============================================================
class TCNRNG:
    def __init__(self):
        self.u_state = np.zeros((TCN_WIN, N_INPUT), dtype=np.float32)
        self.last_y  = np.zeros(N_OUTPUT, dtype=np.float32)
        self.reset()

    def reset(self):
        for t in range(TCN_WIN):
            for i in range(N_INPUT):
                self.u_state[t, i] = 0.01 * (i + 1 + t)
        self.last_y[:] = 0.0

    def step(self):
        # shift window
        self.u_state[:-1] = self.u_state[1:]

        # feedback
        self.u_state[-1, :] = self.last_y[:N_INPUT]

        # forward pass
        h1 = conv1d(self.u_state, W_conv1, B_conv1, TCN_CH1)
        h2 = conv1d(h1,         W_conv2, B_conv2, TCN_CH2)

        y = dense_out(h2[-1])

        # feedback
        self.last_y[:] = y

        # RNG
        raw = srt_taps(y)
        return arx(raw)

# ============================================================
# Example run (NIST‑ready)
# ============================================================
if __name__ == "__main__":
    rng = TCNRNG()

    with open("tcn_rng_output.bin", "wb") as f:
        for _ in range(10000):
            f.write(rng.step().tobytes())

    print("Generated tcn_rng_output.bin")


Generated tcn_rng_output.bin
